# 다양한 LLM API 활용과 모델 대결

<table style="margin: 0; text-align: left; width:100%">
  <tr>
    <th style="text-align:left; width:30%">항목</th>
    <th style="text-align:left">내용</th>
  </tr>
  <tr>
    <td>주제</td>
    <td>여러 LLM API를 동시에 활용하는 다중 모델 대결 시스템 구축</td>
  </tr>
  <tr>
    <td>사용 API</td>
    <td>OpenAI, Anthropic, Google Gemini, DeepSeek, Groq, Ollama</td>
  </tr>
  <tr>
    <td>에이전트 패턴</td>
    <td>Prompt Chaining + Parallelization + Evaluator</td>
  </tr>
  <tr>
    <td>핵심 개념</td>
    <td>API 클라이언트 초기화, OpenAI 호환 엔드포인트, JSON 구조화 출력, LLM-as-Judge</td>
  </tr>
</table>

---

## 학습 목표

1. **다양한 LLM API의 공통점과 차이점**을 이해하고 각각을 초기화하는 방법을 익힌다.
2. **Parallelization 패턴**을 활용해 동일한 질문을 여러 모델에 동시에 전송하는 구조를 구현한다.
3. **LLM-as-Judge(Evaluator) 패턴**으로 다른 LLM의 응답을 자동 평가하는 방법을 배운다.
4. **JSON 구조화 출력**을 요청하고 파싱하는 기법을 실습한다.

---

## 이 노트북에서 사용하는 에이전트 패턴

| 패턴 | 설명 | 이 노트북에서 역할 |
|------|------|------------------|
| **Prompt Chaining** | 한 LLM의 출력이 다음 LLM의 입력이 되는 파이프라인 | 질문 생성 → 답변 수집 → Judge 평가 |
| **Parallelization** | 동일한 작업을 여러 LLM에 병렬로 전송 | 여러 모델이 같은 질문에 독립적으로 답변 |
| **Evaluator** | 한 LLM이 다른 LLM들의 결과를 평가 | GPT Judge가 모든 답변을 채점하고 순위 결정 |

## 전체 시스템 구조

```
┌─────────────────────────────────────────────────────────────┐
│              질문 생성기 (GPT-4o-mini)                       │
│   "LLM 지능을 평가할 수 있는 어려운 한국어 질문 생성"         │
└──────────────────────────┬──────────────────────────────────┘
                           │ 어려운 한국어 질문 1개
                           ▼
┌─────────────────────────────────────────────────────────────┐
│                  Parallelization                             │
│                                                             │
│  ┌─────────┐  ┌─────────┐  ┌─────────┐  ┌─────────┐       │
│  │  GPT    │  │ Claude  │  │ Gemini  │  │DeepSeek │       │
│  │4o-mini  │  │  Haiku  │  │  Flash  │  │  Chat   │       │
│  └────┬────┘  └────┬────┘  └────┬────┘  └────┬────┘       │
│       │            │            │             │             │
│  ┌─────────┐  ┌─────────┐                                  │
│  │  Groq   │  │ Ollama  │                                  │
│  │ Llama   │  │  로컬   │                                  │
│  └────┬────┘  └────┬────┘                                  │
│       │            │                                        │
│       └────────────┴─────────── 답변 집계 ─────────────────┘
│                                                             │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│               Judge (GPT-4o-mini)                            │
│          논리성 · 명확성 · 깊이 기준으로 평가               │
│          JSON 형식으로 순위 반환                             │
└──────────────────────────┬──────────────────────────────────┘
                           │ {"results": ["2", "1", "3", ...]}
                           ▼
                     최종 순위 출력
```

> **Prompt Chaining**: 질문 생성 → 답변 수집 → 평가로 이어지는 파이프라인  
> **Parallelization**: 동일한 질문이 여러 모델에 동시에 전달됨  
> **Evaluator**: Judge LLM이 모든 답변을 비교해 순위를 결정함

In [ ]:
# 필요한 라이브러리 임포트
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display
import concurrent.futures

# .env 파일에서 API 키 로드 (override=True: 이미 설정된 환경변수도 덮어씀)
load_dotenv(override=True)

print("라이브러리 로드 완료!")

In [ ]:
# API 키 존재 여부 확인
# (전체 키를 출력하면 보안 위험 → 앞 몇 글자만 확인)

openai_api_key    = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key    = os.getenv('GOOGLE_API_KEY')
deepseek_api_key  = os.getenv('DEEPSEEK_API_KEY')
groq_api_key      = os.getenv('GROQ_API_KEY')

def check_key(name, key, prefix_len=8):
    if key:
        print(f"✅ {name}: 설정됨 (시작: {key[:prefix_len]}...)")
    else:
        print(f"❌ {name}: 설정되지 않음")

check_key("OpenAI",    openai_api_key)
check_key("Anthropic", anthropic_api_key, 7)
check_key("Google",    google_api_key, 6)
check_key("DeepSeek",  deepseek_api_key, 6)
check_key("Groq",      groq_api_key, 8)

## LLM API 비교: 공통점과 차이점

| API | 클라이언트 클래스 | base_url | 특이사항 |
|-----|-----------------|----------|----------|
| **OpenAI** | `OpenAI()` | 기본값 | 표준 형태 |
| **Anthropic** | `Anthropic()` | 기본값 | `max_tokens` 필수, 응답 구조 다름 |
| **Gemini** | `OpenAI(base_url=...)` | Google 엔드포인트 | OpenAI 호환 |
| **DeepSeek** | `OpenAI(base_url=...)` | DeepSeek 엔드포인트 | OpenAI 호환 |
| **Groq** | `OpenAI(base_url=...)` | Groq 엔드포인트 | OpenAI 호환, 오픈소스 모델 고속 추론 |
| **Ollama** | `OpenAI(base_url=...)` | `localhost:11434` | 로컬 실행, 인터넷 불필요 |

### 응답 구조 비교

```
┌─────────────────────────────────────────────────────┐
│  OpenAI / Gemini / DeepSeek / Groq / Ollama         │
│  (OpenAI 호환 API 공통)                              │
│                                                     │
│  response                                           │
│    └── .choices[0]                                  │
│           └── .message                              │
│                  └── .content  ← 텍스트 응답        │
└─────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────┐
│  Anthropic API (독자 규격)                           │
│                                                     │
│  response                                           │
│    └── .content[0]                                  │
│           └── .text  ← 텍스트 응답                  │
└─────────────────────────────────────────────────────┘
```

> **핵심**: Anthropic을 제외한 대부분의 API는 OpenAI 호환 형식을 사용합니다.  
> base_url만 바꾸면 같은 클라이언트 코드로 다양한 서비스를 이용할 수 있습니다.

In [ ]:
# 각 API 클라이언트 초기화
# API 키가 없는 서비스는 이후 해당 셀에서 오류가 발생할 수 있으니 건너뛰세요.

# 1. OpenAI - 환경변수 OPENAI_API_KEY를 자동으로 읽음
openai_client = OpenAI()

# 2. Anthropic - 환경변수 ANTHROPIC_API_KEY를 자동으로 읽음
claude_client = Anthropic()

# 3. Gemini - OpenAI 호환 엔드포인트 사용 (Google의 OpenAI 호환 레이어)
gemini_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 4. DeepSeek - OpenAI 호환 엔드포인트 사용
deepseek_client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com/v1"
)

# 5. Groq - OpenAI 호환 엔드포인트, 오픈소스 모델을 빠르게 실행
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

# 6. Ollama - 로컬에서 실행 중인 LLM 서버에 연결
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # 로컬이므로 실제 키는 필요없음
)

print("모든 클라이언트 초기화 완료!")

## Step 1: 질문 생성기 (Prompt Chaining 시작)

```
Prompt Chaining 패턴:

  [GPT] → 질문 생성
    │
    └──→ question 변수에 저장
              │
              └──→ 다음 단계(Step 2)의 입력으로 사용
```

**왜 LLM이 질문을 생성하는가?**

- 인간이 매번 창의적인 질문을 만드는 대신, GPT가 자동으로 다양한 난이도의 질문을 생성합니다.
- 이는 **자동화된 벤치마킹 파이프라인**의 핵심 요소입니다.
- 실제 프로덕션에서는 이 패턴으로 LLM 성능을 지속적으로 모니터링합니다.

**프롬프트 설계 포인트:**
- `"질문만 출력하고 설명은 하지 마세요"` → 파싱 가능한 출력 유도
- 한국 사회, 철학, 역사 등 맥락 있는 주제 지정 → 단순 암기보다 추론 능력 측정

In [ ]:
# Step 1: GPT로 어려운 한국어 질문 생성

request = (
    "LLM들의 지능을 평가할 수 있는 도전적이고 심층적인 한국어 질문을 하나 만들어주세요. "
    "한국 사회 이슈, 철학적 딜레마, 역사적 사건의 현대적 의미, "
    "또는 복잡한 윤리적 문제를 다루어도 좋습니다. "
    "질문만 출력하고, 설명이나 답변은 포함하지 마세요."
)

messages = [{"role": "user", "content": request}]

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
)
question = response.choices[0].message.content

print("생성된 질문:")
print("=" * 60)
print(question)

## Step 2: Parallelization (Sectioning) 패턴

```
Parallelization 패턴:

         question (동일한 입력)
            │
    ┌───────┼───────┬───────┬───────┬───────┐
    ▼       ▼       ▼       ▼       ▼       ▼
  GPT    Claude  Gemini  DeepSeek Groq  Ollama
    │       │       │       │       │       │
    ▼       ▼       ▼       ▼       ▼       ▼
 answer1 answer2 answer3 answer4 answer5 answer6
    │       │       │       │       │       │
    └───────┴───────┴───────┴───────┴───────┘
                         │
                    answers 리스트
```

**이 패턴의 장점:**
- 여러 모델의 답변을 비교해 **가장 신뢰할 수 있는 답변** 선택 가능
- 한 모델이 실패해도 다른 모델의 결과 활용 가능 (장애 허용성)
- 각 모델의 강점이 다른 도메인에서 **앙상블 효과** 기대

아래 셀들을 순서대로 실행하면서 각 모델의 응답 스타일 차이를 관찰해보세요.

In [ ]:
# 답변 수집용 리스트 초기화
competitors = []
answers = []

# 질문을 messages 형식으로 변환 (모든 모델에 동일한 형식으로 전달)
messages = [{"role": "user", "content": question}]

print(f"질문: {question}")
print(f"\n총 {len(messages)}개의 메시지를 각 모델에 전송합니다.")

In [ ]:
# --- 참가자 1: OpenAI GPT-4o-mini ---
# OpenAI 표준 API 사용

model_name = "gpt-4o-mini"

response = openai_client.chat.completions.create(
    model=model_name,
    messages=messages
)
answer = response.choices[0].message.content  # OpenAI 응답 구조

display(Markdown(f"### GPT-4o-mini 답변\n\n{answer}"))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# --- 참가자 2: Anthropic Claude Haiku ---
# ⚠️ Anthropic API는 다른 구조!
#   - max_tokens 파라미터가 필수
#   - 응답: response.content[0].text (response.choices[0].message.content 아님)

model_name = "claude-haiku-4-5-20251001"

response = claude_client.messages.create(
    model=model_name,
    messages=messages,
    max_tokens=1000  # Anthropic API에서는 필수 파라미터
)
answer = response.content[0].text  # ← OpenAI와 다른 응답 구조!

display(Markdown(f"### Claude Haiku 답변\n\n{answer}"))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# --- 참가자 3: Google Gemini Flash ---
# OpenAI 호환 엔드포인트 사용 → 코드 구조는 OpenAI와 동일
# 차이점은 클라이언트 초기화 시 base_url을 Google로 지정한 것뿐

model_name = "gemini-2.0-flash"

response = gemini_client.chat.completions.create(
    model=model_name,
    messages=messages
)
answer = response.choices[0].message.content

display(Markdown(f"### Gemini 2.0 Flash 답변\n\n{answer}"))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# --- 참가자 4: DeepSeek Chat ---
# 중국 AI 회사 DeepSeek의 모델
# OpenAI 호환 엔드포인트를 제공하므로 코드 구조 동일

model_name = "deepseek-chat"

response = deepseek_client.chat.completions.create(
    model=model_name,
    messages=messages
)
answer = response.choices[0].message.content

display(Markdown(f"### DeepSeek Chat 답변\n\n{answer}"))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# --- 참가자 5: Groq - Llama 3.3 70B ---
# Groq: 오픈소스 모델을 특수 하드웨어(LPU)로 초고속 추론하는 서비스
# Meta의 Llama 모델을 클라우드에서 무료(또는 저렴하게) 실행 가능

model_name = "llama-3.3-70b-versatile"

response = groq_client.chat.completions.create(
    model=model_name,
    messages=messages
)
answer = response.choices[0].message.content

display(Markdown(f"### Groq Llama 3.3 70B 답변\n\n{answer}"))
competitors.append(model_name)
answers.append(answer)

## Ollama: 로컬에서 실행하는 오픈소스 LLM

```
클라우드 API vs 로컬 Ollama:

클라우드:  [내 코드] ──인터넷──→ [원격 서버의 GPU] ──→ 응답

Ollama:    [내 코드] ──localhost──→ [내 컴퓨터의 Ollama 서버] ──→ 응답
```

**Ollama 장점:**
- 데이터가 외부로 나가지 않음 (프라이버시/보안)
- API 비용 없음
- 인터넷 없이 동작

**Ollama 단점:**
- 로컬 컴퓨터 성능에 따라 속도 차이
- 대형 모델은 메모리 부족 가능

**설치 방법:**
1. https://ollama.com 에서 다운로드 및 설치
2. 터미널에서 `ollama pull llama3.2` 실행으로 모델 다운로드
3. `ollama serve`로 서버 시작 (설치 후 자동 시작되기도 함)
4. http://localhost:11434 접속 시 "Ollama is running" 확인

**주의:** `llama3.3` (43GB)는 가정용 컴퓨터에 너무 큽니다. `llama3.2` (2GB)를 사용하세요.

In [ ]:
# --- 참가자 6: Ollama (로컬 LLM) ---
# Ollama가 설치되지 않은 경우 이 셀은 건너뛰세요.

try:
    # 모델 다운로드 (이미 있으면 건너뜀, 약 2GB)
    !ollama pull llama3.2

    model_name = "llama3.2"

    response = ollama_client.chat.completions.create(
        model=model_name,
        messages=messages
    )
    answer = response.choices[0].message.content

    display(Markdown(f"### Ollama Llama 3.2 답변\n\n{answer}"))
    competitors.append(model_name)
    answers.append(answer)

except Exception as e:
    print(f"Ollama 실행 실패: {e}")
    print("Ollama가 설치되어 있지 않거나 실행 중이지 않습니다. 이 단계를 건너뜁니다.")

In [ ]:
# 지금까지 수집된 답변 요약 확인

print(f"총 {len(competitors)}개 모델의 답변 수집 완료")
print("=" * 60)

for i, (competitor, answer) in enumerate(zip(competitors, answers), 1):
    print(f"\n[참가자 {i}] 모델: {competitor}")
    print("-" * 40)
    # 긴 답변은 앞부분만 출력
    preview = answer[:200] + "..." if len(answer) > 200 else answer
    print(preview)

## Step 3: Judge (Evaluator) 패턴

```
Evaluator 패턴:

  answer1 ┐
  answer2 ├──→ [Judge LLM] ──→ JSON 순위 ──→ 파싱 ──→ 결과 출력
  answer3 │        ↑
  ...     ┘   judge_prompt
              (평가 기준 포함)
```

**LLM-as-Judge 패턴의 핵심:**

1. **구조화된 출력 요청**: `"JSON 형식으로만 답하세요"` → 파싱 가능한 응답 유도
2. **명확한 평가 기준 제시**: 논리성, 명확성, 깊이 등 구체적 기준
3. **참가자 번호 익명화**: 모델 이름 대신 번호 사용 → 편향 방지

**JSON 구조화 출력 요청 팁:**
```
프롬프트에 포함할 것:
  - 정확한 JSON 스키마 예시
  - "마크다운이나 코드 블록 없이 JSON만 출력"
  - 이중 중괄호 이스케이프: {{ }} (f-string 내부)
```

In [ ]:
# Judge 프롬프트 구성

# 모든 답변을 하나의 텍스트로 합치기
together = ""
for index, answer in enumerate(answers):
    together += f"# 참가자 {index + 1}의 답변\n\n"
    together += answer + "\n\n"

# Judge 프롬프트 작성
judge_prompt = f"""당신은 {len(competitors)}개 LLM 모델의 답변을 평가하는 전문 심사위원입니다.

아래 질문에 대해 각 참가자가 답변했습니다:

질문: {question}

평가 기준:
1. 논리성: 주장이 논리적으로 일관되고 근거가 충분한가
2. 명확성: 핵심 내용이 명확하게 전달되는가
3. 깊이: 주제를 심층적으로 분석했는가

다음 JSON 형식으로 순위를 매겨주세요 (가장 좋은 답변부터):
{{"results": ["1위 참가자 번호", "2위 참가자 번호", "3위 참가자 번호", ...]}}

반드시 순수한 JSON만 출력하세요. 코드 블록(```), 마크다운, 추가 설명을 절대 포함하지 마세요.
출력의 첫 글자는 반드시 {{ 이어야 합니다.

각 참가자의 답변:

{together}
"""

print("Judge 프롬프트 구성 완료")
print(f"총 프롬프트 길이: {len(judge_prompt)} 글자")


In [ ]:
# Judge 실행 및 결과 파싱

judge_messages = [{"role": "user", "content": judge_prompt}]

judge_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=judge_messages,
)
result_text = judge_response.choices[0].message.content

print("Judge 원본 응답:")
print(result_text)
print()

# JSON 파싱
try:
    results_dict = json.loads(result_text)
    ranks = results_dict["results"]

    print("=" * 50)
    print("🏆 최종 순위")
    print("=" * 50)

    medals = ["🥇", "🥈", "🥉"]
    for index, result in enumerate(ranks):
        competitor = competitors[int(result) - 1]
        medal = medals[index] if index < 3 else f"{index + 1}위"
        print(f"{medal} {index + 1}위: {competitor}")

except json.JSONDecodeError as e:
    print(f"JSON 파싱 실패: {e}")
    print("Judge가 올바른 JSON을 반환하지 않았습니다. 프롬프트를 수정해보세요.")


## 패턴 분석: 이 노트북에서 사용한 에이전트 패턴

```
┌─────────────────────────────────────────────────────────────┐
│  패턴 1: Prompt Chaining                                    │
│                                                             │
│  [GPT]          [여러 LLM]        [GPT Judge]              │
│  질문 생성  ──→  답변 수집  ──→  평가 및 순위              │
│  (Step 1)       (Step 2)         (Step 3)                  │
│                                                             │
│  각 단계의 출력이 다음 단계의 입력이 되는 파이프라인        │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│  패턴 2: Parallelization (Sectioning)                       │
│                                                             │
│  question ──→ GPT    ──→ answer1                           │
│           ──→ Claude ──→ answer2  (동일 입력,              │
│           ──→ Gemini ──→ answer3   독립적 처리)            │
│           ──→ ...    ──→ ...                               │
│                                                             │
│  같은 작업을 여러 에이전트에게 분산해 결과를 비교          │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│  패턴 3: Evaluator (LLM-as-Judge)                          │
│                                                             │
│  [answer1, answer2, ...]                                   │
│          │                                                  │
│          ▼                                                  │
│    [Judge LLM]  ←── judge_prompt (평가 기준)               │
│          │                                                  │
│          ▼                                                  │
│    JSON 순위 파싱 → 최종 결과                              │
│                                                             │
│  한 LLM이 다른 LLM들의 결과를 객관적으로 평가             │
└─────────────────────────────────────────────────────────────┘
```

## 비즈니스 활용 시나리오 및 요약

### 실제 활용 사례

| 시나리오 | 적용 방법 |
|---------|----------|
| **고객 응대 품질 향상** | 여러 LLM이 답변 생성 → Judge가 최적 답변 선택 후 발송 |
| **콘텐츠 생성** | 여러 모델이 초안 작성 → 품질 기준으로 최우수 선택 |
| **자동 번역 품질 검증** | 여러 번역 모델 결과를 Judge LLM이 평가 |
| **LLM 선택 기준 수집** | 도메인별로 어떤 모델이 우수한지 데이터 축적 |
| **A/B 테스트 자동화** | 새 모델 버전과 기존 버전을 자동으로 비교 |

### 핵심 정리

1. **API 호환성**: 대부분의 LLM이 OpenAI 호환 엔드포인트를 제공 → `base_url`만 변경하면 동일 코드 재사용
2. **Anthropic 예외**: `max_tokens` 필수, `response.content[0].text`로 응답 접근
3. **JSON 구조화 출력**: 프롬프트에 스키마를 명시하고 마크다운 블록 금지 지시 필요
4. **Evaluator 편향 주의**: Judge도 LLM이므로 자사 모델을 선호하는 편향이 있을 수 있음
5. **비용 vs 품질**: 더 강력한 모델을 Judge로 사용할수록 평가 품질 향상

## 연습문제

### 기본
1. **다른 평가 기준 추가**: Judge 프롬프트에 `"한국어 자연스러움"`, `"창의성"` 기준을 추가하고 순위 변화를 관찰해보세요.
2. **모델 변경**: GPT-4o-mini 대신 `gpt-4o`를 Judge로 사용하면 순위가 달라지나요?

### 심화
3. **병렬 API 호출 구현**: `concurrent.futures.ThreadPoolExecutor`를 사용해 모든 모델에 동시에 요청을 보내고 응답 시간을 비교해보세요.

```python
# 힌트: 병렬 호출 구조
def get_answer(args):
    client, model, messages = args
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

tasks = [
    (openai_client, "gpt-4o-mini", messages),
    (gemini_client, "gemini-2.0-flash", messages),
    # ...
]

with concurrent.futures.ThreadPoolExecutor() as executor:
    results = list(executor.map(get_answer, tasks))
```

4. **응답 메타데이터 수집**: 각 모델의 응답 토큰 수(`response.usage.completion_tokens`)와 응답 시간을 측정하고 비교 테이블을 만들어보세요.

5. **Judge 신뢰도 향상**: 같은 답변을 3번 Judge에게 평가시키고, 가장 많이 1위를 차지한 모델을 최종 승자로 결정하는 **다수결 방식**을 구현해보세요.